<a href="https://colab.research.google.com/github/chaitu64/Reddit-AI-Reels-Auto-Post/blob/main/Reddit-AI-Reels-Auto-Post.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas praw
!pip install torch transformers tqdm
!pip install langdetect googletrans==4.0.0-rc1
!pip install pydub
!pip install git+https://github.com/openai/whisper.git
!pip install google-auth-oauthlib==0.4.1 google-auth==1.35.0 google-api-python-client==2.0.2
!pip install moviepy

# Install system packages
!apt-get -qq -y install espeak-ng ffmpeg

In [ ]:
# Import necessary libraries
import praw
import pandas as pd
from datetime import datetime, timedelta
import time
import random

# Setup Reddit API using PRAW
def get_reddit_instance(client_id, client_secret, user_agent):
    """
    Create a Reddit instance using PRAW

    Parameters:
    - client_id: Your Reddit API client ID
    - client_secret: Your Reddit API client secret
    - user_agent: A unique identifier for your app

    Returns:
    - reddit: Authenticated Reddit instance
    """
    try:
        reddit = praw.Reddit(
            client_id=client_id,
            client_secret=client_secret,
            user_agent=user_agent,
            check_for_async=False
        )
        return reddit
    except Exception as e:
        print(f"Authentication failed: {e}")
        return None

# Get yesterday's posts from specified subreddits
def get_yesterdays_posts(reddit, subreddit_names, posts_per_subreddit=10):
    """
    Extract posts from multiple subreddits that were posted yesterday,
    ordered by upvotes

    Parameters:
    - reddit: Authenticated Reddit instance
    - subreddit_names: List of subreddit names to fetch from
    - posts_per_subreddit: Maximum number of posts to return per subreddit

    Returns:
    - DataFrame containing post details sorted by upvotes
    """
    try:
        # Calculate yesterday's date range
        today = datetime.now()
        yesterday = today - timedelta(days=1)
        yesterday_start = datetime(yesterday.year, yesterday.month, yesterday.day, 0, 0, 0)
        yesterday_end = datetime(yesterday.year, yesterday.month, yesterday.day, 23, 59, 59)

        # Convert to Unix timestamp
        yesterday_start_unix = int(yesterday_start.timestamp())
        yesterday_end_unix = int(yesterday_end.timestamp())

        # Create empty lists to store post details
        titles = []
        post_texts = []
        upvotes = []
        comments = []

        # Track the number of posts found from yesterday
        total_yesterday_posts = 0

        # Process each subreddit
        print(f"Collecting posts from yesterday ({yesterday.strftime('%Y-%m-%d')})...")

        for subreddit_name in subreddit_names:
            print(f"Fetching yesterday's posts from r/{subreddit_name}...")
            subreddit = reddit.subreddit(subreddit_name)

            # Get new posts (we need to fetch more to ensure we get enough from yesterday)
            new_posts = subreddit.new(limit=300)  # Increased limit to find more yesterday's posts

            # Keep track of posts from this subreddit
            subreddit_yesterday_posts = []

            # Filter for posts from yesterday
            for post in new_posts:
                # Check if post was created yesterday
                if yesterday_start_unix <= post.created_utc <= yesterday_end_unix:
                    subreddit_yesterday_posts.append(post)

                # Add a small delay to avoid rate limiting
                time.sleep(random.uniform(0.1, 0.3))

            # Sort posts by upvotes
            subreddit_yesterday_posts.sort(key=lambda post: post.score, reverse=True)

            # Take the top posts_per_subreddit
            top_subreddit_posts = subreddit_yesterday_posts[:posts_per_subreddit]

            print(f"Found {len(subreddit_yesterday_posts)} posts from yesterday in r/{subreddit_name}, taking top {len(top_subreddit_posts)}")
            total_yesterday_posts += len(top_subreddit_posts)

            # Extract data from the top posts
            for post in top_subreddit_posts:
                titles.append(post.title)
                post_texts.append(post.selftext)
                upvotes.append(post.score)
                comments.append(post.num_comments)

        # Create DataFrame with collected data
        posts_df = pd.DataFrame({
            'Title': titles,
            'Text': post_texts,
            'Upvotes': upvotes,
            'Comments': comments
        })

        # Sort by upvotes in descending order
        posts_df = posts_df.sort_values(by='Upvotes', ascending=False)

        return posts_df

    except Exception as e:
        print(f"Error getting posts: {e}")
        return None

# Main function to execute the script
def main():
    # Your Reddit API credentials
    client_id = ""  # Replace with your actual client ID
    client_secret = ""  # Replace with your actual client secret

    # Create a specific and unique user agent
    user_agent = "python:com.example.top_posts_collector:v1.0 (by /u/YOUR_REDDIT_USERNAME)"

    # List of subreddits to fetch from
    subreddits = [
        "nosleep",
        "AmItheAsshole",
        "confession",
        "relationships",
        "tifu",
        "pettyrevenge",
        "Ghoststories",
        "stories"
    ]

    # Number of posts to collect per subreddit
    POSTS_PER_SUBREDDIT = 1

    print("Authenticating with Reddit API...")
    reddit = get_reddit_instance(client_id, client_secret, user_agent)

    if reddit is None:
        print("Authentication failed. Please check your credentials.")
        return

    print(f"Extracting yesterday's top {POSTS_PER_SUBREDDIT} posts from each of the specified subreddits...")
    posts_df = get_yesterdays_posts(
        reddit,
        subreddits,
        posts_per_subreddit=POSTS_PER_SUBREDDIT
    )

    if posts_df is not None:
        # Display the results summary
        print(f"Successfully extracted {len(posts_df)} top posts.")

        # Show total number of posts
        print(f"- Total posts: {len(posts_df)}")

        # Show preview of yesterday's top posts by upvotes
        print("\nPreview of yesterday's top posts by upvotes:")
        preview = posts_df[['Title', 'Upvotes', 'Comments']].head(10)
        print(preview)

        # Save to CSV
        filename = f"reddit_posts.csv"

        # Save to CSV with all collected data
        posts_df.to_csv(filename, index=False)

        print(f"\nData saved to {filename}")
    else:
        print("Failed to extract posts.")

# Execute the script
if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import langdetect
from googletrans import Translator
import time
from tqdm import tqdm

# Initialize the translator
translator = Translator()

def detect_language(text):
    """Detect the language of the given text."""
    try:
        if pd.isna(text) or text.strip() == '':
            return 'unknown'
        return langdetect.detect(text)
    except:
        return 'unknown'

def translate_text(text, src_lang):
    """Translate text to English if it's not already in English."""
    if pd.isna(text) or text.strip() == '' or src_lang == 'en' or src_lang == 'unknown':
        return text

    try:
        # Add a small delay to avoid hitting API limits
        time.sleep(0.5)
        translation = translator.translate(text, src=src_lang, dest='en')
        return translation.text
    except Exception as e:
        print(f"Translation error: {e}")
        return text  # Return original text if translation fails

# Load the CSV file
file_path = 'reddit_posts.csv'
df = pd.read_csv(file_path)

# Identify the text column(s)
potential_text_columns = ['Text']
text_columns = [col for col in df.columns if col in potential_text_columns]

if not text_columns:
    print(f"No text columns found. Available columns are: {df.columns.tolist()}")
    # If no predefined column names match, assume the first column with string data is the text column
    for col in df.columns:
        if df[col].dtype == 'object':
            text_columns = [col]
            print(f"Using column '{col}' for text processing")
            break

if not text_columns:
    print("No suitable text columns found!")
else:
    # Add a language detection column
    print("Processing text columns...")

    for col in text_columns:
        print(f"Processing column: {col}")

        # Store original text in a temporary column just for backup
        df['_temp_original'] = df[col].copy()

        # Detect language for each row and store in a language column
        print("Detecting languages...")
        df['text_lang'] = df[col].apply(detect_language)

        # Count languages detected
        lang_counts = df['text_lang'].value_counts()
        print("Languages detected:")
        print(lang_counts)

        # Translate non-English text and replace original text
        print("Translating non-English text...")
        non_english_rows = df['text_lang'] != 'en'
        total_to_translate = non_english_rows.sum()

        if total_to_translate > 0:
            print(f"Found {total_to_translate} rows with non-English text to translate")

            # Translate only non-English rows and update original column
            for idx in tqdm(df[non_english_rows].index):
                src_lang = df.at[idx, 'text_lang']
                original_text = df.at[idx, col]
                translated_text = translate_text(original_text, src_lang)
                df.at[idx, col] = translated_text

            print(f"Translated {total_to_translate} rows in column '{col}'")
        else:
            print("No non-English text found, no translation needed")

    # Remove the temporary column
    df.drop('_temp_original', axis=1, inplace=True)

    # Save the result with translated text
    output_file = 'translated_file.csv'
    df.to_csv(output_file, index=False)
    print(f"Saved results to {output_file}")

In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline
import re
import nltk
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Download necessary NLTK data
import os

# Create a directory for NLTK data in the current working directory
nltk_data_dir = os.path.join(os.getcwd(), 'nltk_data')
os.makedirs(nltk_data_dir, exist_ok=True)

# Download the required NLTK resources to the local directory
nltk.download('punkt', download_dir=nltk_data_dir, quiet=True)
nltk.data.path.append(nltk_data_dir)  # Add our directory to NLTK's search path

def count_words(text):
    """Count the number of words in a text."""
    if pd.isna(text) or text == "":
        return 0

    text = str(text)

    # Use a simple but robust tokenization approach
    text = re.sub(r'([.,!?;:])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    words = [w for w in text.split() if w.strip()]
    return len(words)

def clean_text(text):
    """Clean the text by removing URLs, extra spaces, etc."""
    if pd.isna(text) or text == "":
        return ""

    # Convert to string if it's not already
    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+', '', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def adjust_summary_length(text, target_min=250, target_max=300):
    """Adjust the summary to be between target_min and target_max words."""
    word_count = count_words(text)

    # If already in the desired range, return as is
    if target_min <= word_count <= target_max:
        return text

    # If too long, truncate
    if word_count > target_max:
        words = text.split()
        return ' '.join(words[:target_max])

    # For summaries that are too short, we'll keep the original
    # (this shouldn't happen with our parameters but added for robustness)
    return text

def main():
    # Specify input file path as requested
    input_file = 'translated_file.csv'
    output_file = 'final_text.csv'

    print(f"Loading CSV data from {input_file}...")
    try:
        # Load the CSV file
        df = pd.read_csv(input_file)
        print(f"Successfully loaded CSV with {len(df)} rows")

        # Print the column names to help with debugging
        print(f"Columns in the CSV: {df.columns.tolist()}")

        # Determine which column contains the post content
        content_columns = [col for col in df.columns if any(term in col.lower()
                          for term in ['text', 'content', 'post', 'body', 'selftext'])]

        if not content_columns:
            print("Could not find a column that likely contains post content.")
            print("Please provide the name of the column containing post text:")
            content_column = input()
        else:
            content_column = content_columns[0]
            print(f"Using column '{content_column}' for post content")

        # Clean the text data
        print("Cleaning text data...")
        df['cleaned_text'] = df[content_column].apply(clean_text)

        # Count words in each post
        print("Counting words in each post...")
        df['word_count'] = df['cleaned_text'].apply(count_words)

        # Initialize summarization only if needed
        long_posts = len(df[df['word_count'] > 300])
        if long_posts > 0:
            print(f"Found {long_posts} posts longer than 300 words that need summarization")

            try:
                # Initialize the summarization model
                print("Loading summarization model...")
                summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

                # Process each row with minimal messaging
                summarized_texts = []
                total_posts = len(df)

                for idx, row in df.iterrows():
                    if idx % 20 == 0:  # Show progress less frequently
                        print(f"Processing item {idx+1}/{total_posts} ({(idx+1)/total_posts*100:.1f}%)")

                    # If text is already under 200 words, keep it as is
                    if row['word_count'] <= 200:
                        summarized_texts.append(row['cleaned_text'])
                        continue

                    # For longer texts, we need to summarize to between 250-300 words
                    try:
                        text = row['cleaned_text']

                        # Set summarization parameters to target our desired range
                        # We aim for 200-230 words from the model, then will adjust if needed
                        target_length = 230
                        max_length = target_length
                        min_length = 200

                        # Attempt summarization with error handling
                        try:
                            summary = summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)
                            summary_text = summary[0]['summary_text']

                            # Adjust the summary to be within our target range
                            summary_text = adjust_summary_length(summary_text, 250, 300)
                        except:
                            # Fall back to truncation if summarization fails
                            words = text.split()
                            summary_text = ' '.join(words[:275])  # Aim for middle of range

                        summarized_texts.append(summary_text)
                    except:
                        # Silently fail and use truncation as a last resort
                        words = row['cleaned_text'].split()
                        summarized_texts.append(' '.join(words[:275]))

                # Assign the results back to the dataframe
                df['summarized_text'] = summarized_texts

                # Double-check word counts to ensure we're in the right range
                df['final_word_count'] = df['summarized_text'].apply(count_words)

                # Report the results
                in_range = df[(df['word_count'] > 300) & (250 <= df['final_word_count']) & (df['final_word_count'] <= 300)]
                print(f"Successfully summarized {len(in_range)} of {long_posts} long posts to 250-300 words")

                # Make any final adjustments if needed
                df['summarized_text'] = df.apply(
                    lambda row: row['summarized_text'] if row['word_count'] <= 300 or (250 <= row['final_word_count'] <= 300)
                    else adjust_summary_length(row['summarized_text'], 250, 300),
                    axis=1
                )

                print("Summarization complete!")

            except Exception as e:
                print(f"Summarization error: {e}")
                print("Falling back to simple truncation for all posts")
                # Fallback: truncate long posts to 275 words (middle of our target range)
                df['summarized_text'] = df.apply(
                    lambda row: row['cleaned_text'] if row['word_count'] <= 300
                    else ' '.join(row['cleaned_text'].split()[:275]),
                    axis=1
                )
        else:
            print("No posts require summarization, using original text")
            df['summarized_text'] = df['cleaned_text']

        # Make sure we only keep the final summarized_text column
        # (remove any duplicate attempts from the code)
        if 'summarized_text' in df.columns:
            # Keep only essential columns in final output
            result_df = df[[content_column, 'summarized_text']]

            # Save results to specified output file
            print(f"Saving results to {output_file}...")
            result_df.to_csv(output_file, index=False)

            print(f"Process complete! Saved {len(df)} processed posts.")

            # Print summary statistics
            if 'final_word_count' in df.columns:
                long_summarized = df[df['word_count'] > 300]
                avg_length = long_summarized['final_word_count'].mean()
                print(f"Average summary length: {avg_length:.1f} words")
                print(f"Summaries in target range (250-300): {len(long_summarized[(250 <= long_summarized['final_word_count']) & (long_summarized['final_word_count'] <= 300)])} of {len(long_summarized)}")
        else:
            print("Error: Failed to generate summarized text column")

    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import torch
import soundfile as sf
from IPython.display import display, Audio
import os
import numpy as np
from pydub import AudioSegment
import re

# Voice configuration - only two voices as requested
VOICES = {
    "female": "af_heart",  # Female voice
    "male": "am_adam"      # Male voice
}

# Create output directory if it doesn't exist
os.makedirs("story_audio", exist_ok=True)
os.makedirs("story_audio/temp_segments", exist_ok=True)

# Function to install required packages if they're missing
def install_required_packages():
    try:
        import pip
        required_packages = ["pydub", "numpy", "soundfile", "pandas"]
        for package in required_packages:
            try:
                __import__(package)
            except ImportError:
                print(f"Installing required package: {package}")
                pip.main(["install", package])

        # Try to install kokoro automatically
        try:
            pip.main(["install", "kokoro"])
            print("Kokoro installed successfully")
        except:
            print("Kokoro installation failed")

    except Exception as e:
        print(f"Failed to install required packages: {e}")
        print("Please manually install: pydub, numpy, soundfile, pandas")

# Install required packages
install_required_packages()

# Automatically load the CSV file - use default if not found
try:
    csv_file = 'final_text.csv'
    df = pd.read_csv(csv_file)
    print(f"Successfully loaded CSV with {len(df)} entries")
except FileNotFoundError:
    print(f"CSV file {csv_file} not found. Creating sample data for demonstration.")
    # Create sample data for demonstration
    df = pd.DataFrame({
        'summarized_text_translated': [
            "Once upon a time in a small village, there lived a young girl who discovered she could talk to animals.",
            "The spaceship landed quietly in the desert. Its occupants were not what the scientists expected.",
            "The old bookstore held a secret passage that only opened during a full moon.",
            "Every night at midnight, the clocktower would play a melody that no one remembered placing there.",
            "The detective knew the answer was hidden somewhere in the painting."
        ]
    })
    print("Created sample data for demonstration")

# Identify the column containing the story text
text_column = None
possible_columns = ['summarized_text']

for col in possible_columns:
    if col in df.columns:
        text_column = col
        break

if text_column is None:
    print(f"Could not find text column. Available columns: {df.columns.tolist()}")
    text_column = df.columns[0]  # Use the first column as fallback
    print(f"Using '{text_column}' as fallback")

print(f"Using column '{text_column}' for text data")

# Function to determine if a story has female or male narration
def determine_gender(text):
    """
    Simple heuristic to determine if a story is likely narrated by a female or male character.
    This is a basic implementation and could be improved with NLP techniques.
    """
    # Look for common female pronouns/indicators
    female_indicators = ['she', 'her', 'herself', 'woman', 'girl', 'mother', 'daughter', 'sister', 'aunt']
    # Look for common male pronouns/indicators
    male_indicators = ['he', 'him', 'himself', 'man', 'boy', 'father', 'son', 'brother', 'uncle']

    # Convert to lowercase for case-insensitive matching
    text_lower = text.lower()

    # Count occurrences
    female_count = sum([text_lower.count(' ' + word + ' ') for word in female_indicators])
    male_count = sum([text_lower.count(' ' + word + ' ') for word in male_indicators])

    # Return gender based on counts
    if female_count > male_count:
        return "female"
    else:
        return "male"  # Default to male if equal or more male indicators

# Process all entries in the dataset
num_entries = len(df)
print(f"Processing all {num_entries} entries from the dataset")

# Determine voice for each story based on content
story_voices = {}
for i in range(num_entries):
    text = df[text_column].iloc[i]
    gender = determine_gender(text)
    story_voices[i] = VOICES[gender]
    print(f"Story {i+1}: Detected as {gender} narration, using voice {VOICES[gender]}")

# Function for TTS conversion with segment joining
def text_to_speech(text, voice, output_path, sample_rate=24000):
    """
    Convert text to speech using the specified voice and join all segments.

    Args:
        text: The text to convert to speech
        voice: The voice to use
        output_path: Where to save the final joined audio
        sample_rate: Sample rate of the audio (default: 24000)
    """
    print(f"Converting text using voice: {voice}")
    print(f"Text: {text[:100]}..." if len(text) > 100 else f"Text: {text}")

    # Create a temp directory for segments
    temp_dir = os.path.join(os.path.dirname(output_path), "temp_segments")
    os.makedirs(temp_dir, exist_ok=True)

    try:
        # Initialize Kokoro pipeline if using it
        from kokoro import KPipeline
        pipeline = KPipeline(lang_code='a')

        # Generate audio segments
        all_audio_data = []
        segment_paths = []

        # Use the generator to create segments
        generator = pipeline(text, voice=voice)

        for i, (gs, ps, audio) in enumerate(generator):
            # Save each segment temporarily
            segment_path = os.path.join(temp_dir, f"{os.path.basename(output_path).split('.')[0]}_seg_{i}.wav")
            segment_paths.append(segment_path)
            sf.write(segment_path, audio, sample_rate)
            all_audio_data.append(audio)
            print(f"Generated segment {i+1}")

        # Join all audio segments into one file
        join_audio_segments(segment_paths, output_path, sample_rate)
        print(f"All segments joined and saved to: {output_path}")

        # Clean up temporary segment files
        for segment_path in segment_paths:
            if os.path.exists(segment_path):
                os.remove(segment_path)

        return True

    except ImportError:
        print("Kokoro library not found. Using simulation mode.")
        # For demonstration purposes (when library is not available)
        with open(output_path.replace('.wav', '.txt'), 'w') as f:
            f.write(f"Voice: {voice}\n\n{text}")
        return True
    except Exception as e:
        print(f"Error in text-to-speech conversion: {e}")
        return False


def join_audio_segments(segment_paths, output_path, sample_rate=24000):
    """
    Join multiple audio segments into a single file.

    Args:
        segment_paths: List of paths to audio segment files
        output_path: Path to save the joined audio
        sample_rate: Audio sample rate
    """
    # Method 1: Using pydub (more robust, handles different formats)
    try:
        # Check if we have any segments
        if not segment_paths:
            print("No segments to join")
            return False

        # Join with pydub
        combined = AudioSegment.empty()
        for path in segment_paths:
            sound = AudioSegment.from_file(path)
            combined += sound

        # Export the combined audio
        combined.export(output_path, format="wav")
        return True

    except Exception as e:
        print(f"Error joining with pydub: {e}")
        # Fallback to numpy method
        try:
            # Method 2: Using numpy and soundfile (simpler, but less flexible)
            all_audio = []
            for path in segment_paths:
                audio_data, _ = sf.read(path)
                all_audio.append(audio_data)

            # Concatenate all audio data
            combined_audio = np.concatenate(all_audio)

            # Write combined audio to file
            sf.write(output_path, combined_audio, sample_rate)
            return True

        except Exception as e2:
            print(f"Error joining with numpy: {e2}")
            return False

# Process the entries
for i in range(num_entries):
    text = df[text_column].iloc[i]
    voice = story_voices[i]

    if not isinstance(text, str) or len(text.strip()) == 0:
        print(f"Skipping empty or non-string entry at index {i}")
        continue

    print(f"\n--- Processing Story {i+1}/{num_entries} ---")
    output_file = f"story_audio/story_{i+1}_{voice}.wav"

    # Convert long text into manageable chunks if needed
    if len(text) > 1000:
        # Split text into sentences first
        sentences = re.split('(?<=[.!?])\s+', text)

        # Group sentences into chunks of reasonable size
        chunks = []
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) < 1000:
                current_chunk += " " + sentence if current_chunk else sentence
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = sentence

        # Add the last chunk if it exists
        if current_chunk:
            chunks.append(current_chunk)

        print(f"Text split into {len(chunks)} chunks for processing")

        # Process each chunk separately and collect segment paths
        segment_paths = []
        for j, chunk in enumerate(chunks):
            chunk_path = f"story_audio/temp_chunk_{i+1}_{j+1}.wav"
            success = text_to_speech(chunk, voice, chunk_path)
            if success:
                segment_paths.append(chunk_path)
                print(f"Successfully processed chunk {j+1}/{len(chunks)} for Story {i+1}")
            else:
                print(f"Failed to process chunk {j+1}/{len(chunks)} for Story {i+1}")

        # Join all chunks into one final story audio
        if segment_paths:
            join_audio_segments(segment_paths, output_file)
            print(f"Successfully joined all chunks for Story {i+1} with voice {voice}")

            # Clean up temporary chunk files
            for path in segment_paths:
                if os.path.exists(path):
                    os.remove(path)
    else:
        # Process text directly if it's not too long
        success = text_to_speech(text, voice, output_file)
        if success:
            print(f"Successfully processed Story {i+1} with voice {voice}")
        else:
            print(f"Failed to process Story {i+1}")

# Print voice assignments for reference
print("\nVoice assignments summary:")
gender_counts = {"female": 0, "male": 0}
for i in range(num_entries):
    voice = story_voices[i]
    if voice == VOICES["female"]:
        gender_counts["female"] += 1
    else:
        gender_counts["male"] += 1

print(f"Female voice stories: {gender_counts['female']}")
print(f"Male voice stories: {gender_counts['male']}")

print("\nProcessing complete!")

In [ ]:
import os
import subprocess
import sys
import random

def check_ytdlp_installation():
    """Check if yt-dlp is installed, and install it if not."""
    try:
        subprocess.run(["yt-dlp", "--version"], capture_output=True, text=True)
        print("yt-dlp is already installed.")
    except FileNotFoundError:
        print("yt-dlp not found. Installing...")
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "yt-dlp"], check=True)
            print("yt-dlp installed successfully.")
        except subprocess.CalledProcessError:
            print("Failed to install yt-dlp. Please install it manually.")
            sys.exit(1)

def create_videos_folder():
    """Create a 'videos' folder if it doesn't exist."""
    if not os.path.exists("videos"):
        os.makedirs("videos")
        print("Created 'videos' folder.")
    else:
        print("'videos' folder already exists.")

def download_video(url, title):
    """Download video at the best resolution and save to the videos folder."""
    print(f"\nDownloading {title}...")

    # Format: videos/%(title)s.%(ext)s
    cmd = [
        "yt-dlp",
        "-f", "bestvideo+bestaudio/best",  # Best video+audio or best available if combined
        "-o", f"videos/{title}.%(ext)s",
        "--no-playlist",
        url
    ]

    try:
        subprocess.run(cmd, check=True)
        print(f"Successfully downloaded {title}!")
    except subprocess.CalledProcessError as e:
        print(f"Error downloading {title}: {e}")

def main():
    # Combined list of original and new video URLs with unique titles
    videos = {
        "1": {"title": "GTA_Original", "url": "https://youtu.be/P3b3AidldmA?si=6BwoN-aV35FIxUiQ"},
        "2": {"title": "Minecraft_Original", "url": "https://youtu.be/cjxxE2gwEVg?si=MjUwnuDCClngt5BI"},
        "3": {"title": "Subway_Surfers_Original", "url": "https://youtu.be/LZzwQrXIDEM?si=7y2BpxZfNrmkdo4A"},
        "4": {"title": "GTA_New", "url": "https://youtu.be/tCBOhczn6Ok?si=YbqK5BM_m8Lmv7i4"},
        "5": {"title": "Subway_Surfers_New", "url": "https://youtu.be/2SWW4U4TKOA?si=6hBkUexEIjC1ldg5"}
    }

    # Check for yt-dlp installation
    check_ytdlp_installation()

    # Create videos folder
    create_videos_folder()

    # Randomly select a video from all available options
    choice = random.choice(list(videos.keys()))
    selected_video = videos[choice]

    # Download the selected video
    download_video(selected_video["url"], selected_video["title"])
    print("\nDownload completed!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import subprocess
import json
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def get_media_info(file_path):
    """Get duration and other metadata for a media file using FFprobe"""
    try:
        cmd = [
            'ffprobe',
            '-v', 'quiet',
            '-print_format', 'json',
            '-show_format',
            '-show_streams',
            file_path
        ]

        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        info = json.loads(result.stdout)

        # Get duration from format section
        duration = float(info['format']['duration'])
        return {'duration': duration, 'info': info}

    except (subprocess.SubprocessError, KeyError, json.JSONDecodeError) as e:
        print(f"Error getting media info for {file_path}: {str(e)}")
        return None

def sync_single_video(audio_path, video_path, output_path, start_position=0):
    """
    Process a single video-audio pair using FFmpeg (much faster than MoviePy)

    Args:
        audio_path: Path to the audio file
        video_path: Path to the video file
        output_path: Path where the output will be saved
        start_position: Start position in the video (in seconds)

    Returns:
        Success status (True/False) and an error message if applicable
    """
    try:
        # Get audio duration
        audio_info = get_media_info(audio_path)
        if not audio_info:
            return False, f"Failed to get audio info for {audio_path}"

        audio_duration = audio_info['duration']

        # Create the FFmpeg command
        cmd = [
            'ffmpeg',
            '-y',  # Overwrite output files without asking
            '-ss', str(start_position),  # Start position in video
            '-i', video_path,  # Input video
            '-i', audio_path,  # Input audio
            '-map', '0:v',  # Use video from first input
            '-map', '1:a',  # Use audio from second input
            '-c:v', 'copy',  # Copy video codec (no re-encoding, very fast)
            '-c:a', 'aac',  # Convert audio to AAC
            '-shortest',  # Finish encoding when the shortest input stream ends
            '-t', str(audio_duration),  # Limit duration to audio duration
            output_path
        ]

        # Run FFmpeg
        subprocess.run(cmd, check=True, capture_output=True)
        return True, None

    except subprocess.SubprocessError as e:
        error_message = f"FFmpeg error: {str(e)}"
        if hasattr(e, 'stderr') and e.stderr:
            error_message += f"\nDetails: {e.stderr.decode('utf-8')}"
        return False, error_message
    except Exception as e:
        return False, f"Error: {str(e)}"

def sync_videos_with_audios_ffmpeg(audio_dir, video_dir, output_dir, max_workers=4):
    """
    Sync audio files with video content sequentially using FFmpeg (much faster than MoviePy).

    Args:
        audio_dir: Directory containing audio files
        video_dir: Directory containing video files
        output_dir: Directory where output files will be saved
        max_workers: Maximum number of parallel processes
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Get all audio files
    audio_extensions = ('.mp3', '.wav', '.ogg', '.m4a', '.aac')
    audio_files = []

    for ext in audio_extensions:
        audio_files.extend(glob.glob(os.path.join(audio_dir, f'*{ext}')))
        audio_files.extend(glob.glob(os.path.join(audio_dir, f'*{ext.lower()}')))

    # Remove duplicates and sort
    audio_files = sorted(list(set(audio_files)))

    if not audio_files:
        print(f"No audio files found in {audio_dir}")
        return

    # Get all video files
    video_extensions = ('.mp4', '.webm', '.mov', '.mkv', '.avi')
    video_files = []

    for ext in video_extensions:
        video_files.extend(glob.glob(os.path.join(video_dir, f'*{ext}')))
        video_files.extend(glob.glob(os.path.join(video_dir, f'*{ext.lower()}')))

    # Remove duplicates and sort
    video_files = sorted(list(set(video_files)))

    if not video_files:
        print(f"No video files found in {video_dir}")
        return

    print(f"Found {len(audio_files)} audio files and {len(video_files)} video files")

    # Get video durations (to avoid repeatedly calling FFprobe)
    video_durations = {}
    print("Getting video durations...")
    for video_path in tqdm(video_files):
        info = get_media_info(video_path)
        if info:
            video_durations[video_path] = info['duration']

    # Initialize variables to track current position in videos
    current_video_index = 0
    current_position_in_video = 0

    # Create a list of tasks
    tasks = []

    # Process each audio file
    for i, audio_path in enumerate(audio_files):
        audio_name = os.path.basename(audio_path)
        print(f"\nPreparing audio {i+1}/{len(audio_files)}: {audio_name}")

        # Get audio duration
        audio_info = get_media_info(audio_path)
        if not audio_info:
            print(f"Skipping {audio_name}: Cannot determine duration")
            continue

        audio_duration = audio_info['duration']

        # Get current video
        video_path = video_files[current_video_index]
        video_name = os.path.basename(video_path)

        # Check if we have cached the duration
        if video_path not in video_durations:
            video_info = get_media_info(video_path)
            if not video_info:
                print(f"Skipping {video_name}: Cannot determine duration")
                continue
            video_durations[video_path] = video_info['duration']

        video_duration = video_durations[video_path]
        remaining_video_duration = video_duration - current_position_in_video

        # Check if we need to move to the next video
        if remaining_video_duration < audio_duration:
            print(f"Not enough space in current video (only {remaining_video_duration:.2f}s left). Moving to next video.")

            # Move to the next video
            current_video_index = (current_video_index + 1) % len(video_files)
            current_position_in_video = 0

            # Get the new video
            video_path = video_files[current_video_index]
            video_name = os.path.basename(video_path)

            # Recalculate remaining duration for the new video
            if video_path not in video_durations:
                video_info = get_media_info(video_path)
                if not video_info:
                    print(f"Skipping {video_name}: Cannot determine duration")
                    continue
                video_durations[video_path] = video_info['duration']

        # Generate output filename
        output_name = f"story_{i+1}.mp4"
        output_path = os.path.join(output_dir, output_name)

        # Create task
        tasks.append({
            'audio_path': audio_path,
            'video_path': video_path,
            'output_path': output_path,
            'start_position': current_position_in_video,
            'audio_duration': audio_duration
        })

        # Update the position for the next audio
        current_position_in_video += audio_duration

        # Check if we've reached the end of the current video
        if current_position_in_video >= video_durations[video_path] - 2:  # Less than 2 seconds remaining
            current_video_index = (current_video_index + 1) % len(video_files)
            current_position_in_video = 0

    # Process tasks in parallel
    print(f"\nProcessing {len(tasks)} tasks with {max_workers} workers...")

    def process_task(task):
        print(f"Processing: {os.path.basename(task['audio_path'])} + {os.path.basename(task['video_path'])}")
        success, error = sync_single_video(
            task['audio_path'],
            task['video_path'],
            task['output_path'],
            task['start_position']
        )

        if success:
            return f"Successfully created {os.path.basename(task['output_path'])}"
        else:
            return f"Failed to process {os.path.basename(task['audio_path'])}: {error}"

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(process_task, tasks), total=len(tasks)))

    for result in results:
        print(result)

    print("\nAll processing complete!")


# Example usage
if __name__ == "__main__":
    # Update these paths to match your environment
    audio_dir = '/content/story_audio'
    video_dir = '/content/videos'
    output_dir = 'shorts'

    # Choose which implementation to use

    # Option 1: Create individual videos for each audio
    # Set max_workers based on your CPU cores (typically 4-8 is good)
    sync_videos_with_audios_ffmpeg(audio_dir, video_dir, output_dir, max_workers=4)


In [ ]:
import os
import subprocess
import tempfile
import shutil
import json
import re
from datetime import timedelta
from IPython.display import HTML, display
import tqdm
import torch
import numpy as np
import wave
import time
from pydub import AudioSegment
import requests
import concurrent.futures

# Use faster_whisper which leverages CTranslate2 for optimized Whisper inference
try:
    from faster_whisper import WhisperModel
except ImportError:
    print("Installing faster_whisper for optimized performance...")
    !pip install faster_whisper
    from faster_whisper import WhisperModel

def add_word_highlighted_subtitles(input_folder="shorts", output_folder="subtitled",
                                  font="Roboto", device="auto",
                                  max_workers=2):
    """
    Add word-by-word highlighted subtitles to videos in the input folder.
    Subtitles appear line by line in the middle of the screen, with the current word highlighted in yellow.
    Uses the specified font (default: Roboto).
    Videos are saved in the output folder with the same filenames.

    Parameters:
    - input_folder: Folder containing video files
    - output_folder: Folder for subtitled output videos
    - font: Font to use for subtitles
    - device: Device to run inference on ('cpu', 'cuda', 'auto')
    - max_workers: Maximum number of videos to process in parallel
    """
    start_time = time.time()

    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Check if input folder exists
    if not os.path.exists(input_folder):
        print(f"Input folder '{input_folder}' not found. Creating it...")
        os.makedirs(input_folder)
        print(f"Please upload your videos to the '{input_folder}' folder and run this function again.")
        return

    # Find all video files in the input folder
    video_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.mp4', '.mov', '.avi'))]

    if not video_files:
        print(f"No video files found in '{input_folder}' folder. Please upload videos first.")
        return

    print(f"Found {len(video_files)} videos to process.")

    # Create a temporary work directory
    work_dir = os.path.join(tempfile.gettempdir(), "subtitle_work")
    os.makedirs(work_dir, exist_ok=True)

    # Download font - always use Roboto
    font_path = ensure_font(work_dir, "Roboto")

    # If font download failed, continue with a fallback message
    if font_path == "Arial":
        print("⚠️ Using Arial as fallback font since font download failed")

    # Always use the base model
    model_size = "base"
    print(f"Loading {model_size} Whisper model using faster_whisper...")

    # Automatically choose appropriate compute type based on device
    compute_type = "float16" if device == "cuda" or (device == "auto" and torch.cuda.is_available()) else "int8"
    model = WhisperModel(model_size, device=device, compute_type=compute_type)
    # The WhisperModel class doesn't expose device directly
    used_device = "GPU" if device == "cuda" or (device == "auto" and torch.cuda.is_available()) else "CPU"
    print(f"Model loaded successfully on {used_device} using {compute_type}!")

    # Process videos in parallel for better overall throughput
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create a dictionary to store future objects with video file names as keys
        future_to_video = {
            executor.submit(
                process_single_video,
                video_file,
                input_folder,
                output_folder,
                work_dir,
                model,
                font_path
            ): video_file for video_file in video_files
        }

        # Process completed futures as they finish
        for future in concurrent.futures.as_completed(future_to_video):
            video_file = future_to_video[future]
            try:
                result_status, processing_time = future.result()
                print(f"{video_file}: {result_status} (took {processing_time:.1f} seconds)")
            except Exception as e:
                print(f"{video_file}: Failed - {str(e)}")

    # Clean up temporary files
    shutil.rmtree(work_dir)

    total_time = time.time() - start_time
    print(f"\nAll videos processed in {total_time:.2f} seconds!")
    print(f"Results saved to '{output_folder}' folder.")
    print("You can download the subtitled videos from the file browser.")

def process_single_video(video_file, input_folder, output_folder, work_dir, model, font_path):
    """Process a single video file with subtitles"""
    video_start_time = time.time()

    input_path = os.path.join(input_folder, video_file)
    base_name = os.path.splitext(video_file)[0]
    output_path = os.path.join(output_folder, f"{base_name}.mp4")

    try:
        # Create a subfolder for this video's temporary files
        video_temp_dir = os.path.join(work_dir, base_name)
        os.makedirs(video_temp_dir, exist_ok=True)

        # 1. Extract audio efficiently using pydub - much faster than running full ffmpeg
        audio_path = os.path.join(video_temp_dir, f"{base_name}.wav")
        extract_audio_efficient(input_path, audio_path)

        # 2. Generate word-level transcription with faster_whisper
        result = transcribe_with_faster_whisper(model, audio_path)

        # Save transcript for reference
        transcript_json = os.path.join(video_temp_dir, f"{base_name}.json")
        with open(transcript_json, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2)

        # 3. Create ASS subtitle file with word highlighting
        ass_path = os.path.join(video_temp_dir, f"{base_name}.ass")
        create_word_highlighting_ass(result, ass_path, font_path)

        # 4. Burn subtitles into video (using optimized settings)
        burn_subtitles_efficiently(input_path, ass_path, output_path)

        processing_time = time.time() - video_start_time
        return "✅ Success", processing_time

    except Exception as e:
        if os.path.exists(output_path):
            os.remove(output_path)
        return f"❌ Failed: {str(e)}", time.time() - video_start_time

def ensure_font(work_dir, font_name="Roboto"):
    """Download and install the specified font if needed"""
    import requests
    from pathlib import Path

    # Define font directory and path
    font_dir = os.path.join(work_dir, "fonts")
    os.makedirs(font_dir, exist_ok=True)
    font_path = os.path.join(font_dir, f"{font_name}-Regular.ttf")

    # Common Google Font URLs
    font_urls = {
        "Roboto": "https://fonts.gstatic.com/s/roboto/v30/KFOmCnqEu92Fr1Mu4mxKKTU1Kg.woff2",
        "OpenSans": "https://fonts.gstatic.com/s/opensans/v34/memSYaGs126MiZpBA-UvWbX2vVnXBbObj2OVZyOOSr4dVJWUgsjZ0B4gaVI.woff2",
        "Montserrat": "https://fonts.gstatic.com/s/montserrat/v25/JTUHjIg1_i6t8kCHKm4532VJOt5-QNFgpCtr6Hw5aXo.woff2"
    }

    # Always use Roboto
    font_name = "Roboto"
    font_path = os.path.join(font_dir, f"{font_name}-Regular.ttf")

    # Download if not exists
    if not os.path.exists(font_path):
        print(f"  Downloading {font_name} font...")

        try:
            response = requests.get(font_urls[font_name])

            if response.status_code == 200:
                with open(font_path, 'wb') as f:
                    f.write(response.content)
                print(f"  {font_name} font downloaded successfully")
            else:
                print(f"  Failed to download {font_name} font: {response.status_code}")
                # Fall back to Arial if download fails
                font_path = "Arial"
        except Exception as e:
            print(f"  Error downloading font: {str(e)}")
            font_path = "Arial"

    return font_path

def extract_audio_efficient(input_video, output_audio):
    """Extract audio efficiently - optimized for speed"""
    try:
        # Fast extraction with ffmpeg - extract at 16kHz mono directly
        cmd = [
            'ffmpeg', '-y', '-i', input_video,
            '-vn', '-acodec', 'pcm_s16le',
            '-ar', '16000', '-ac', '1',
            '-loglevel', 'error',
            output_audio
        ]
        subprocess.run(cmd, check=True, stderr=subprocess.PIPE)
        return
    except (subprocess.CalledProcessError, FileNotFoundError):
        # If ffmpeg direct approach fails, try pydub
        try:
            audio = AudioSegment.from_file(input_video)
            audio = audio.set_channels(1).set_frame_rate(16000)
            audio.export(output_audio, format="wav")
        except Exception as e:
            # Last resort, try most basic ffmpeg command
            cmd = [
                'ffmpeg', '-y', '-i', input_video,
                '-vn', output_audio
            ]
            subprocess.run(cmd, check=True)

def transcribe_with_faster_whisper(model, audio_path):
    """Transcribe audio using faster_whisper with word-level timestamps"""
    # Use optimized settings
    segments, info = model.transcribe(
        audio_path,
        word_timestamps=True,
        vad_filter=True,         # Voice activity detection to skip silence
        vad_parameters=dict(min_silence_duration_ms=500),  # Minimize processing of silence
        language="en",           # Force English for faster recognition if content is in English
        beam_size=3,             # Smaller beam size for speed
        best_of=1,               # Only keep best result
        temperature=0.0,         # Greedy decoding for faster results
    )

    # Process segments into a format similar to whisper's output for compatibility
    result = {"segments": []}

    for segment in segments:
        segment_dict = {
            "id": segment.id,
            "start": segment.start,
            "end": segment.end,
            "text": segment.text.strip(),
            "words": []
        }

        # Add word-level information
        if segment.words:
            for word in segment.words:
                segment_dict["words"].append({
                    "word": word.word,
                    "start": word.start,
                    "end": word.end,
                    "probability": word.probability
                })

        result["segments"].append(segment_dict)

    return result

def format_timestamp(seconds):
    """Convert seconds to ASS timestamp format (h:mm:ss.cc)"""
    td = timedelta(seconds=seconds)
    hours, remainder = divmod(td.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    centiseconds = int(td.microseconds / 10000)
    return f"{hours}:{minutes:02d}:{seconds:02d}.{centiseconds:02d}"

def create_word_highlighting_ass(result, ass_path, font_name):
    """
    Create ASS subtitle file with individual word highlighting
    - Shows one line at a time in the middle of the screen
    - Current word is highlighted in yellow
    - Other words are white
    - Uses specified font
    - Maximum 4 words per line
    """
    with open(ass_path, 'w', encoding='utf-8') as f:
        # Write header with styles
        f.write(
            "[Script Info]\n"
            "Title: Word Highlighting Subtitles\n"
            "ScriptType: v4.00+\n"
            "WrapStyle: 0\n"  # Force no line wrapping
            "ScaledBorderAndShadow: yes\n"
            "PlayResX: 1080\n"  # Vertical video typical dimensions
            "PlayResY: 1920\n\n"
            "[V4+ Styles]\n"
            "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n"
            # Default style - white text, centered in middle (Alignment=5 for middle-center)
            f"Style: Default,{font_name},55,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,2.5,1.5,5,10,10,10,1\n"
            # Highlighted style - yellow text
            f"Style: Highlight,{font_name},55,&H0000FFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,2.5,1.5,5,10,10,10,1\n\n"
            "[Events]\n"
            "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n"
        )

        # Split segments into shorter phrases for single-line display
        processed_segments = []

        for segment in result["segments"]:
            if "words" not in segment or not segment["words"]:
                continue

            words = segment["words"]

            # Group words into smaller phrases (MAX 4 words per phrase)
            current_phrase = []
            current_start = None

            for word_data in words:
                word = word_data["word"].strip()
                if not word:
                    continue

                if current_start is None:
                    current_start = word_data["start"]

                current_phrase.append(word_data)

                # After collecting max words or at the end, create a phrase
                if len(current_phrase) >= 4:  # Changed from 6 to 4 words max
                    phrase_end = current_phrase[-1]["end"]
                    processed_segments.append({
                        "start": current_start,
                        "end": phrase_end,
                        "words": current_phrase
                    })
                    current_phrase = []
                    current_start = None

            # Add any remaining words as a final phrase
            if current_phrase:
                phrase_end = current_phrase[-1]["end"]
                processed_segments.append({
                    "start": current_start,
                    "end": phrase_end,
                    "words": current_phrase
                })

        # Process each phrase as a single subtitle line
        for phrase in processed_segments:
            phrase_words = phrase["words"]
            full_text = ' '.join([w["word"].strip() for w in phrase_words if w["word"].strip()])

            # Generate the base subtitle line that displays the full text in white
            phrase_start = format_timestamp(phrase["start"])
            phrase_end = format_timestamp(phrase["end"])

            # Write the base white text line with forced single-line
            clean_text = full_text.replace('\\N', ' ').replace('\n', ' ')
            f.write(f"Dialogue: 0,{phrase_start},{phrase_end},Default,,0,0,0,,{clean_text}\n")

            # Now add individual animations for each word
            for word_data in phrase_words:
                word = word_data["word"].strip()
                if not word:
                    continue

                word_start = word_data["start"]
                word_end = word_data["end"]

                # Find position of this word in the clean text
                word_position = clean_text.find(word)
                if word_position == -1:
                    continue

                # Create timestamps
                word_start_stamp = format_timestamp(word_start)
                word_end_stamp = format_timestamp(word_end)

                # Create an override that highlights just this word
                before_text = clean_text[:word_position]
                after_text = clean_text[word_position + len(word):]

                # Write the highlight effect - this overrides just during the word timing
                highlighted_text = f"{before_text}{{\\c&H00FFFF&}}{word}{{\\c&HFFFFFF&}}{after_text}"
                f.write(f"Dialogue: 1,{word_start_stamp},{word_end_stamp},Default,,0,0,0,,{highlighted_text}\n")

def burn_subtitles_efficiently(input_video, subs_file, output_video):
    """Burn subtitles into video using ffmpeg with optimized settings for speed"""
    try:
        # Get video dimensions to determine if we need to scale
        probe_cmd = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0',
            '-show_entries', 'stream=width,height', '-of', 'json',
            input_video
        ]

        try:
            probe_output = subprocess.check_output(probe_cmd, universal_newlines=True)
            video_info = json.loads(probe_output)
            width = int(video_info['streams'][0]['width'])
            height = int(video_info['streams'][0]['height'])

            # Use hardware acceleration if available
            if torch.cuda.is_available():
                hw_accel = '-hwaccel', 'cuda'
                encoder = 'h264_nvenc'
            else:
                hw_accel = []
                encoder = 'libx264'

            # Optimize encoding settings based on video size
            if width * height > 1920 * 1080:
                # For large videos, downsample to 1080p for speed
                scale_filter = f"scale=-1:min(1080,{height})"
                vf = f"{scale_filter},ass={subs_file}"
                preset = 'veryfast'
                crf = '28'  # Lower quality for speed
            else:
                # For smaller videos, keep original resolution
                vf = f"ass={subs_file}"
                preset = 'faster'
                crf = '26'  # Balanced quality/speed

            cmd = [
                'ffmpeg', '-y'] + list(hw_accel) + ['-i', input_video,
                '-vf', vf,
                '-c:a', 'aac', '-ar', '44100',  # Simpler audio settings
                '-c:v', encoder,
                '-preset', preset,
                '-crf', crf,
                '-movflags', '+faststart',  # Optimize for web playback
                output_video
            ]

            subprocess.run(cmd, check=True, stderr=subprocess.PIPE)

        except (json.JSONDecodeError, subprocess.CalledProcessError, KeyError):
            # If probe fails, use safe default settings
            cmd = [
                'ffmpeg', '-y', '-i', input_video,
                '-vf', f"ass={subs_file}",
                '-c:a', 'copy',
                '-c:v', 'libx264',
                '-preset', 'ultrafast',  # Fastest preset
                '-crf', '28',
                output_video
            ]
            subprocess.run(cmd, check=True, stderr=subprocess.PIPE)

    except Exception as e:
        # Final fallback with minimal settings
        cmd = [
            'ffmpeg', '-y', '-i', input_video,
            '-vf', f"subtitles={subs_file}",  # Try subtitles filter instead of ass
            '-c:a', 'copy',
            '-c:v', 'libx264',
            '-preset', 'ultrafast',
            '-crf', '30',  # Lower quality for maximum speed
            output_video
        ]
        subprocess.run(cmd, check=True)

# Function to install additional required packages
def ensure_dependencies():
    try:
        import faster_whisper
        import pydub
    except ImportError:
        print("Installing required packages...")
        !pip install -q faster_whisper pydub

# Install dependencies if needed
ensure_dependencies()

# Example usage
if __name__ == "__main__":
    print("Word-Highlighted Subtitles Generator")
    print("===================================")
    print("Using base Whisper model for optimal balance of speed and accuracy.")

    # Determine device
    device = "auto"  # Will use CUDA if available

    # Determine parallelism based on system resources
    import multiprocessing
    max_workers = min(multiprocessing.cpu_count() // 2, 4)
    if max_workers < 1:
        max_workers = 1

    add_word_highlighted_subtitles(device=device, max_workers=max_workers)

In [ ]:
import os
import csv
import json
import pandas as pd
from datetime import datetime
import random
import re

def generate_title(caption, max_length=60):
    """Generate an engaging title from the caption"""
    # Extract first sentence or first 100 characters
    first_sentence = re.split(r'[.!?]', caption)[0].strip()

    if len(first_sentence) <= max_length:
        return first_sentence
    else:
        # If first sentence is too long, take first part and add ellipsis
        return first_sentence[:max_length-3] + "..."

def generate_description(caption, max_length=300):
    """Generate a description from the caption"""
    # Remove any URLs or special characters
    clean_caption = re.sub(r'http\S+', '', caption)
    clean_caption = re.sub(r'[^\w\s.,!?-]', '', clean_caption)

    # Truncate if needed
    if len(clean_caption) <= max_length:
        return clean_caption
    else:
        return clean_caption[:max_length-3] + "..."

def generate_hashtags(caption, count=5):
    """Generate relevant hashtags from the caption"""
    # List of common hashtags for shorts
    common_hashtags = [
        "#shorts", "#viral", "#trending", "#fyp", "#foryoupage",
        "#story", "#storytelling", "#shortsvideo", "#ytshorts"
    ]

    # Extract potential hashtag words (nouns, adjectives, etc)
    words = re.findall(r'\b[A-Za-z]{5,}\b', caption)

    # Create potential hashtags from these words
    potential_hashtags = ["#" + word.lower() for word in words if len(word) >= 5]

    # Combine both lists, remove duplicates, and select requested count
    all_hashtags = list(set(common_hashtags + potential_hashtags))
    selected_hashtags = random.sample(all_hashtags, min(count, len(all_hashtags)))

    return " ".join(selected_hashtags)

def is_mature_content(text):
    """Basic check for mature content"""
    mature_keywords = [
        "sex", "nude", "porn", "explicit", "adult", "xxx", "nsfw",
        "18+", "mature", "violence", "gore", "blood", "kill"
    ]

    text_lower = text.lower()
    return any(keyword in text_lower for keyword in mature_keywords)

def main():
    # Paths
    input_csv = "final_text.csv"
    output_json = "video_metadata.json"

    print("Starting improved CSV parser for ALL stories...")

    # Check if input file exists
    if not os.path.exists(input_csv):
        print(f"Error: {input_csv} not found.")
        return

    stories = []

    # Try multiple approaches to read the CSV correctly
    try:
        # First attempt: Try standard CSV reading
        print("Attempt 1: Standard CSV reading...")
        df = pd.read_csv(input_csv)

        # Get all text from all columns (ignoring header)
        for _, row in df.iterrows():
            for col in df.columns:
                value = str(row[col]).strip()
                if value and value.lower() != 'nan':
                    stories.append(value)

        print(f"Found {len(stories)} potential stories across all columns")

        # If we didn't find any, try a different approach
        if not stories:
            print("Attempt 2: Reading raw lines and skipping header...")
            with open(input_csv, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            if len(lines) > 1:  # At least header plus one line
                for line in lines[1:]:  # Skip header
                    if line.strip():
                        stories.append(line.strip())

            print(f"Found {len(stories)} non-empty lines (excluding header)")

        # No longer limiting to 6 stories
        print(f"Using all {len(stories)} stories found")

    except Exception as e:
        print(f"Error reading CSV: {e}")
        print("Attempting simple text file reading...")

        # Last resort: Basic text file reading
        try:
            with open(input_csv, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            if len(lines) > 1:  # At least header plus one line
                stories = [line.strip() for line in lines[1:] if line.strip()]  # Skip header, use all stories
                print(f"Found {len(stories)} non-empty lines as fallback")

        except Exception as e2:
            print(f"Error reading as text: {e2}")
            return

    # Container for metadata
    video_data = []

    # Process each story
    for index, caption in enumerate(stories):
        try:
            # Generate metadata
            video_number = index + 1
            title = generate_title(caption)
            description = generate_description(caption)
            hashtags = generate_hashtags(caption)
            language = "en"  # Default to English
            is_18_plus = is_mature_content(caption)
            creation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # Create metadata entry
            metadata = {
                "video_file": f"story_{video_number}",
                "title": title,
                "description": description,
                "hashtags": hashtags,
                "language": language,
                "is_18_plus": is_18_plus,
                "creation_date": creation_date,
                "original_caption": caption[:100] + "..." if len(caption) > 100 else caption
            }

            video_data.append(metadata)
            print(f"Processed story {video_number}: {title[:30]}...")

        except Exception as e:
            print(f"Error processing story {index+1}: {e}")

    # Save metadata to JSON
    try:
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(video_data, f, ensure_ascii=False, indent=4)
        print(f"Successfully saved metadata to {output_json}")
    except Exception as e:
        print(f"Error saving metadata: {e}")

    # Also create a CSV version for compatibility
    try:
        metadata_df = pd.DataFrame(video_data)
        metadata_df.to_csv("video_metadata.csv", index=False)
        print(f"Successfully saved metadata to video_metadata.csv")
    except Exception as e:
        print(f"Error saving CSV metadata: {e}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import time
import google.auth
import google.auth.transport.requests
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow, Flow
from google.auth.transport.requests import Request
import glob

class YouTubeUploader:
    """Class to handle YouTube video uploads with metadata from JSON"""

    def __init__(self, client_secrets_file):
        """Initialize the uploader with client secrets file"""
        self.client_secrets_file = client_secrets_file
        self.api_service_name = "youtube"
        self.api_version = "v3"
        self.scopes = ["https://www.googleapis.com/auth/youtube.upload"]
        self.youtube = None

    def authenticate(self):
        """Authenticate with YouTube API with improved error handling"""
        creds = None

        # Check if token.json exists (for reusing credentials)
        if os.path.exists('token.json'):
            try:
                creds = Credentials.from_authorized_user_info(
                    json.load(open('token.json')), self.scopes)
                print("Found existing credentials, checking validity...")
            except Exception as e:
                print(f"Error loading existing token: {str(e)}")
                creds = None

        # If credentials don't exist or are invalid, get new ones
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                try:
                    print("Refreshing expired credentials...")
                    creds.refresh(Request())
                except Exception as e:
                    print(f"Error refreshing token: {str(e)}")
                    creds = None

            # If we still need new credentials
            if not creds or not creds.valid:
                print("Need to obtain new credentials...")

                # Try the local server approach first
                try:
                    print("Attempting authentication with browser redirect...")
                    print("If a browser doesn't open automatically, please manually visit the URL that will be displayed")

                    flow = InstalledAppFlow.from_client_secrets_file(
                        self.client_secrets_file, self.scopes)
                    creds = flow.run_local_server(
                        port=8080,
                        success_message="Authentication successful! You can close this tab now."
                    )
                    print("✅ Browser authentication successful!")

                # If local server approach fails, fall back to device flow
                except Exception as e:
                    print(f"Browser redirect authentication failed: {str(e)}")
                    print("Falling back to device flow authentication...")

                    try:
                        flow = Flow.from_client_secrets_file(
                            self.client_secrets_file,
                            scopes=self.scopes,
                            redirect_uri="urn:ietf:wg:oauth:2.0:oob"
                        )
                        auth_url, _ = flow.authorization_url(prompt='consent')

                        print("\n------------------------------------------------------")
                        print(f"Please visit this URL to authenticate: {auth_url}")
                        print("------------------------------------------------------\n")

                        code = input("Enter the authorization code: ")
                        flow.fetch_token(code=code)
                        creds = flow.credentials
                        print("✅ Device flow authentication successful!")

                    except Exception as nested_e:
                        print(f"Device flow authentication also failed: {str(nested_e)}")
                        print("Authentication failed with both methods. Please check your client_secrets.json file.")
                        return False

            # Save credentials for future use
            if creds and creds.valid:
                with open('token.json', 'w') as token:
                    token.write(creds.to_json())
                print("Credentials saved for future use.")

        # Build the YouTube API client
        if creds and creds.valid:
            self.youtube = build(self.api_service_name, self.api_version, credentials=creds)
            print("Authentication successful! YouTube API client ready.")
            return True
        else:
            print("Authentication failed. Unable to create YouTube API client.")
            return False

    def upload_video(self, video_path, metadata):
        """Upload a video with its metadata to YouTube"""
        if not self.youtube:
            print("Not authenticated. Please authenticate first.")
            return False

        # Process video privacy status (default to private for safety)
        privacy_status = metadata.get("privacy_status", "public")  # Get from metadata or default to private

        # Create YouTube upload request
        request_body = {
            "snippet": {
                "title": metadata["title"],
                "description": metadata["description"],
                "tags": metadata["hashtags"].split() if metadata.get("hashtags") else []
            },
            "status": {
                "privacyStatus": privacy_status,
                "selfDeclaredMadeForKids": False
            }
        }

        # Add category ID if provided
        if metadata.get("category_id"):
            request_body["snippet"]["categoryId"] = metadata["category_id"]

        # Create a media file upload object with chunk size optimized
        media = MediaFileUpload(
            video_path,
            chunksize=1024*1024,
            resumable=True
        )

        # Create and execute the upload request
        upload_request = self.youtube.videos().insert(
            part="snippet,status",
            body=request_body,
            media_body=media
        )

        # Execute the request with progress reporting
        print(f"Uploading {os.path.basename(video_path)}...")
        response = None
        last_progress = 0

        while response is None:
            try:
                status, response = upload_request.next_chunk()
                if status:
                    current_progress = int(status.progress() * 100)
                    if current_progress > last_progress + 4:  # Only print every 5% to avoid spam
                        print(f"Uploaded {current_progress}%")
                        last_progress = current_progress
            except Exception as e:
                print(f"Error during upload: {str(e)}")
                print("Retrying in 5 seconds...")
                time.sleep(5)
                continue

        print(f"Upload complete! Video ID: {response['id']}")
        print(f"Video URL: https://youtu.be/{response['id']}")
        return response['id']


def main():
    """Main function to run the YouTube uploader"""
    print("YouTube Video Uploader Tool")
    print("==========================")

    # Path to client secrets file
    client_secrets_file = "client_secrets.json"  # Changed from client_secret.json to match your code

    # Check if the client secrets file exists
    if not os.path.exists(client_secrets_file):
        print(f"❌ Error: {client_secrets_file} not found.")
        print("Please make sure you have your Google API credentials file in the current directory.")
        return

    # Path to the folder containing videos
    videos_folder = "subtitled"  # Adjust if needed
    if not os.path.exists(videos_folder):
        print(f"⚠️ Warning: Videos folder '{videos_folder}' not found. Using current directory.")
        videos_folder = "."

    # Find all video files
    video_files = []
    for ext in ['*.mp4', '*.mov', '*.avi', '*.mkv', '*.webm']:
        video_files.extend(glob.glob(os.path.join(videos_folder, ext)))

    if not video_files:
        print(f"❌ No video files found in '{videos_folder}'.")
        return

    print(f"\nFound {len(video_files)} video files.")

    # Check if metadata file exists
    metadata_file = "video_metadata.json"
    if os.path.exists(metadata_file):
        try:
            with open(metadata_file, 'r') as f:
                metadata_list = json.load(f)
            print(f"Loaded {len(metadata_list)} metadata entries from {metadata_file}")
        except Exception as e:
            print(f"❌ Error loading metadata file: {str(e)}")
            return
    else:
        # If no metadata file exists, generate basic metadata for all videos
        print(f"⚠️ No metadata file found at {metadata_file}")
        print("Generating basic metadata for all videos...")
        metadata_list = []
        for video_path in video_files:
            video_name = os.path.basename(video_path)
            base_name = os.path.splitext(video_name)[0]
            metadata_list.append({
                "video_file": video_name,
                "title": f"{base_name}",
                "description": f"Auto-uploaded video: {base_name}",
                "hashtags": "#video #autoupload",
                "privacy_status": "public"
            })

        # Save the generated metadata
        with open(metadata_file, 'w') as f:
            json.dump(metadata_list, f, indent=2)
        print(f"Created basic metadata file at {metadata_file}")

    # Check if we have metadata for all videos
    video_names = [os.path.basename(v) for v in video_files]
    metadata_video_names = [m.get("video_file", "") for m in metadata_list]

    # Find videos without metadata
    missing_metadata = []
    for video_name in video_names:
        if not any(video_name in meta_name for meta_name in metadata_video_names):
            missing_metadata.append(video_name)

    # Add basic metadata for videos without entries
    if missing_metadata:
        print(f"\nFound {len(missing_metadata)} videos without metadata entries. Adding basic metadata...")
        for video_name in missing_metadata:
            base_name = os.path.splitext(video_name)[0]
            metadata_list.append({
                "video_file": video_name,
                "title": f"{base_name}",
                "description": f"Auto-uploaded video: {base_name}",
                "hashtags": "#video #autoupload",
                "privacy_status": "public"
            })

        # Update the metadata file
        with open(metadata_file, 'w') as f:
            json.dump(metadata_list, f, indent=2)
        print(f"Updated metadata file with {len(missing_metadata)} new entries")

    # Create uploader instance and authenticate
    uploader = YouTubeUploader(client_secrets_file)
    if not uploader.authenticate():
        print("❌ Authentication failed. Cannot proceed with uploads.")
        return

    # Dictionary to store results
    upload_results = []

    # Track which videos were already processed
    processed_videos = set()

    # Upload each video with its metadata
    for metadata in metadata_list:
        video_file_name = metadata.get("video_file")
        if not video_file_name:
            print("\n❌ Error: Missing 'video_file' in metadata entry")
            upload_results.append({
                "video_file": "unknown",
                "status": "Failed",
                "error": "Missing video_file in metadata"
            })
            continue

        # Find matching video file in the videos folder
        matching_files = [f for f in video_files if video_file_name in os.path.basename(f)]

        if matching_files:
            video_path = matching_files[0]
            video_base_name = os.path.basename(video_path)

            # Skip if already processed
            if video_base_name in processed_videos:
                print(f"\nSkipping duplicate metadata entry for: {video_file_name}")
                continue

            processed_videos.add(video_base_name)

            print(f"\nProcessing: {video_file_name}")
            print(f"Found matching video: {os.path.basename(video_path)}")

            # Upload video
            try:
                video_id = uploader.upload_video(video_path, metadata)
                upload_results.append({
                    "video_file": video_file_name,
                    "status": "Success",
                    "video_id": video_id,
                    "youtube_url": f"https://youtu.be/{video_id}"
                })

                # Wait a bit between uploads to avoid quota issues
                print("Waiting 10 seconds before next upload to avoid quota issues...")
                time.sleep(10)

            except Exception as e:
                print(f"Error uploading {video_file_name}: {str(e)}")
                upload_results.append({
                    "video_file": video_file_name,
                    "status": "Failed",
                    "error": str(e)
                })
        else:
            print(f"\nNo matching video found for: {video_file_name}")
            upload_results.append({
                "video_file": video_file_name,
                "status": "Failed",
                "error": "No matching video file found"
            })

    # Check for any unprocessed videos
    unprocessed_videos = set(os.path.basename(v) for v in video_files) - processed_videos

    if unprocessed_videos:
        print(f"\nFound {len(unprocessed_videos)} videos that weren't matched with metadata.")
        print("Uploading these videos with auto-generated metadata...")

        for video_name in unprocessed_videos:
            video_path = [v for v in video_files if os.path.basename(v) == video_name][0]
            base_name = os.path.splitext(video_name)[0]

            # Create basic metadata
            auto_metadata = {
                "video_file": video_name,
                "title": f"{base_name}",
                "description": f"Auto-uploaded video: {base_name}",
                "hashtags": "#video #autoupload",
                "privacy_status": "public"
            }

            # Upload video
            try:
                print(f"\nProcessing unmatched video: {video_name}")
                video_id = uploader.upload_video(video_path, auto_metadata)
                upload_results.append({
                    "video_file": video_name,
                    "status": "Success",
                    "video_id": video_id,
                    "youtube_url": f"https://youtu.be/{video_id}"
                })

                # Wait a bit between uploads to avoid quota issues
                print("Waiting 10 seconds before next upload to avoid quota issues...")
                time.sleep(10)

            except Exception as e:
                print(f"Error uploading {video_name}: {str(e)}")
                upload_results.append({
                    "video_file": video_name,
                    "status": "Failed",
                    "error": str(e)
                })

    # Print summary
    print("\n====== Upload Summary ======")
    success_count = sum(1 for r in upload_results if r["status"] == "Success")
    fail_count = len(upload_results) - success_count

    print(f"Total: {len(upload_results)} videos")
    print(f"Successful: {success_count}")
    print(f"Failed: {fail_count}")
    print("\nDetails:")

    for result in upload_results:
        status = result["status"]
        if status == "Success":
            print(f"✅ {result['video_file']} → {result['youtube_url']}")
        else:
            print(f"❌ {result['video_file']} → {result['error']}")

    # Save results to file
    with open('upload_results.json', 'w') as f:
        json.dump(upload_results, f, indent=2)

    print("\nResults saved to upload_results.json")

    if success_count > 0:
        print("\n🎉 Some videos were successfully uploaded to YouTube!")
    else:
        print("\n❌ No videos were successfully uploaded. Please check the errors above.")


# Run the script
if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\nUpload process interrupted by user.")
    except Exception as e:
        print(f"\n\nUnexpected error: {str(e)}")
        print("Please check your configuration and try again.")

In [ ]:
import os
import glob
import time
import logging
import sys

# Install required packages if not already installed
try:
    from instagrapi import Client
    from instagrapi.exceptions import LoginRequired
except ImportError:
    print("Installing required packages...")
    # Install specific versions to ensure compatibility
    os.system(f"{sys.executable} -m pip install pydantic==1.10.8")
    os.system(f"{sys.executable} -m pip install instagrapi==2.0.0")
    from instagrapi import Client
    from instagrapi.exceptions import LoginRequired

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("instagram_upload.log"),
        logging.StreamHandler()
    ]
)

class InstagramDirectUploader:
    def __init__(self, username, password, session_file="session.json"):
        """
        Initialize the Instagram uploader with credentials.

        Parameters:
        - username: Your Instagram username
        - password: Your Instagram password
        - session_file: File to save login session (to avoid frequent logins)
        """
        self.username = username
        self.password = password
        self.session_file = session_file
        self.client = Client()
        self.login()

        # Predefined Lamborghini Aventador description
        self.lamborghini_description = """🚀 **Lamborghini Aventador** – where raw power meets sculpted beauty. The Aventador is not just a car; it's a statement. With its iconic **V12 engine**, aggressive aerodynamics, and **signature scissor doors**, it turns every street into a runway.
Crafted in Italy with precision and passion, the Aventador pushes the limits of performance and style. Whether you're revving through the city or posting up for the perfect shot, this machine demands attention.
✨ Popular among celebrities, influencers, and car enthusiasts worldwide, the Aventador is a **symbol of luxury, speed, and success**

#lamborghini #aventador #supercar #luxurycars #v12 #carlifestyle #dreamcar #exoticcars #hypercar #carsofinstagram #supercars #luxurylifestyle #italiandesign #carspotting #automotivephotography #carphotography #luxurylife #carbonfiber #sportscar #fastcars #lamborghiniaventador #bullsquad #carenthusiast #racecar #automotiveluxury #italiancar #supercarlifestyle #cargram #carswithoutlimits"""

    def login(self):
        """
        Login to Instagram, with session handling to reduce login frequency.
        """
        try:
            # Try to load session if it exists
            if os.path.exists(self.session_file):
                logging.info("Loading session from file")
                try:
                    self.client.load_settings(self.session_file)
                    self.client.get_timeline_feed()  # Test if session is valid
                    logging.info(f"Logged in as {self.username} using session file")
                    return
                except LoginRequired:
                    logging.info("Session expired, logging in with password")
                except Exception as e:
                    logging.error(f"Error loading session: {e}")

            # If no session or session expired, login with username and password
            logging.info(f"Logging in as {self.username}")

            # Use delay_range to comply with Instagram rate limits
            self.client.login(self.username, self.password, relogin=True)

            # Save session for future use
            self.client.dump_settings(self.session_file)
            logging.info("Session saved to file")

        except Exception as e:
            logging.error(f"Login failed: {e}")
            raise

    def get_video_files(self, videos_folder):
        """
        Gets all video files from the specified folder.
        Returns a list of video file paths.
        """
        video_extensions = ['.mp4', '.mov']  # Instagram primarily supports these formats
        video_files = []

        # Check if folder exists
        if not os.path.exists(videos_folder):
            logging.error(f"Videos folder does not exist: {videos_folder}")
            return []

        for ext in video_extensions:
            video_files.extend(glob.glob(os.path.join(videos_folder, f'*{ext}')))

        logging.info(f"Found {len(video_files)} videos in {videos_folder}")
        return video_files

    def upload_video(self, video_path, is_reel=False):
        """
        Uploads a video to Instagram with Lamborghini description.

        Parameters:
        - video_path: Path to the video file
        - is_reel: Boolean indicating whether to upload as a reel (default: False)

        Returns:
        - Instagram media ID if successful, None otherwise
        """
        try:
            # Verify file exists
            if not os.path.exists(video_path):
                logging.error(f"Video file does not exist: {video_path}")
                return None

            filename = os.path.basename(video_path)
            logging.info(f"Starting upload for {filename}")

            # Use the predefined Lamborghini description
            caption = self.lamborghini_description

            # Upload based on type (reel vs regular post)
            media = None

            if is_reel:
                logging.info("Uploading as a Reel")
                media = self.client.clip_upload(
                    video_path,
                    caption=caption
                )
            else:
                logging.info("Uploading as a regular video post")
                media = self.client.video_upload(
                    video_path,
                    caption=caption
                )

            logging.info(f"Successfully uploaded {filename} with media ID: {media.id}")

            # Add short delay to avoid rate limiting
            time.sleep(2)

            return media.id

        except Exception as e:
            logging.error(f"Error uploading {video_path}: {str(e)}")
            return None

    def upload_videos(self, videos_folder, as_reels=False):
        """
        Main function to upload videos to Instagram with Lamborghini description.

        Parameters:
        - videos_folder: Folder containing videos to upload
        - as_reels: Boolean indicating whether to upload videos as reels
        """
        # Get video files
        video_files = self.get_video_files(videos_folder)

        if not video_files:
            logging.warning("No video files found to upload")
            return []

        logging.info(f"Preparing to upload {len(video_files)} videos")

        # Upload each video
        uploaded_videos = []
        for video_path in video_files:
            try:
                media_id = self.upload_video(video_path, is_reel=as_reels)
                if media_id:
                    uploaded_videos.append((video_path, media_id))

                # Important: Add delay between uploads to avoid rate limiting
                # Instagram is sensitive to rapid sequential uploads
                delay = 60  # Increased from 30 to 60 seconds for better rate limit compliance
                logging.info(f"Waiting {delay} seconds before next upload to avoid rate limits...")
                time.sleep(delay)

            except Exception as e:
                logging.error(f"Error processing {video_path}: {str(e)}")

        logging.info(f"Uploaded {len(uploaded_videos)} videos successfully")
        return uploaded_videos

# Example usage
if __name__ == "__main__":
    # Replace with your actual Instagram credentials
    username = ""  # Replace with your actual username (removed for security)
    password = ""  # Replace with your actual password (removed for security)

    # Set your videos folder
    videos_folder = "/content/subtitled"  # Update this to your actual folder path

    # Setting to upload all videos as reels by default
    upload_as_reels = True  # Always upload as reels

    # Create uploader instance
    uploader = InstagramDirectUploader(username, password)

    # Upload videos
    uploaded_videos = uploader.upload_videos(videos_folder, as_reels=upload_as_reels)

    # Print results summary
    print("\n=== UPLOAD SUMMARY ===")
    print(f"Total videos uploaded: {len(uploaded_videos)}")
    for video_path, media_id in uploaded_videos:
        print(f"✓ {os.path.basename(video_path)} -> Instagram ID: {media_id}")